# Entity-Resolved Knowledge Graph RAG with DSPy (v4)

This notebook builds a **graph-augmented RAG** system on entity-resolved data from Senzing, using **DSPy** for structured LLM interaction with chain-of-thought reasoning. It combines the Anthropic and OpenAI DSPy implementations from `08_dspy_anthropic_rag_v4.ipynb` and `08__dspy_openai_rag_v4.ipynb` into a single notebook with a selectable LLM provider.

**Pipeline:**
1. Connect to Senzing and export resolved entities
2. Build the True Combined Graph (entities + records + relationships)
3. Load the embedding model and pre-created vectors from LanceDB
4. Configure DSPy with both Anthropic (Claude) and OpenAI (GPT) language models
5. Define the DSPy Signature and Module (ChainOfThought)
6. Query with a selectable `provider` parameter

In [1]:
import os
import json
from pathlib import Path
import grpc
import networkx as nx
from sentence_transformers import SentenceTransformer
import lancedb
from senzing import SzEngine, SzError, SzEngineFlags
from senzing_grpc import SzAbstractFactoryGrpc
import dspy

print("All imports successful")

All imports successful


## Connect to Senzing and Export Entities

Opens a gRPC connection to the Senzing engine and streams the complete entity report, requesting the raw JSON for each source record alongside the resolved entity data. This is the same export used in notebook 04 and provides the data for building both graph structures.

In [2]:
SENZING_HOST = os.getenv('SENZING_GRPC_HOST', 'senzing')
SENZING_PORT = os.getenv('SENZING_GRPC_PORT', '8261')

print(f"Connecting to Senzing at {SENZING_HOST}:{SENZING_PORT}")

# Create gRPC channel and engine
grpc_url = f"{SENZING_HOST}:{SENZING_PORT}"
grpc_channel = grpc.insecure_channel(grpc_url)
sz_abstract_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_abstract_factory.create_engine()

print("Connected to Senzing successfully")

# Export all resolved entities
print("\nExporting entity data from Senzing with full details...")

entities = []
export_handle = sz_engine.export_json_entity_report(
    flags=SzEngineFlags.SZ_EXPORT_INCLUDE_ALL_ENTITIES |
          SzEngineFlags.SZ_ENTITY_INCLUDE_RECORD_JSON_DATA |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ALL_RELATIONS |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ENTITY_NAME |
          SzEngineFlags.SZ_ENTITY_INCLUDE_RELATED_MATCHING_INFO
)

count = 0
while True:
    try:
        entity_json = sz_engine.fetch_next(export_handle)
        if not entity_json:
            break
        entities.append(json.loads(entity_json))
        count += 1
        if count % 50 == 0:
            print(f"  Exported {count} entities...", end='\r')
    except StopIteration:
        break

sz_engine.close_export_report(export_handle)
print(f"\nExported {len(entities)} entities total")

with_rels = sum(1 for e in entities if e.get('RELATED_ENTITIES'))
print(f"Entities with relationships: {with_rels}")

Connecting to Senzing at senzing:8261
Connected to Senzing successfully

Exporting entity data from Senzing with full details...
  Exported 150 entities...
Exported 196 entities total
Entities with relationships: 189


## Build True Combined Graph (Entities + Records + Relationships)

Constructs the two-layer graph from notebook 04: entity nodes (large shapes) represent resolved entities, record nodes (small dots) represent original source records. Gray dashed edges show which records were merged into each entity, red solid edges show business relationships between entities.

In [3]:
G_true_combined = nx.Graph()

entity_info = {}
entities_added = 0
records_added = 0

for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    records = entity_data.get('RECORDS', [])
    
    if not records:
        continue
    
    first_record = records[0]
    json_data = first_record.get('JSON_DATA', {})
    
    # v4 format: extract type and name from FEATURES
    record_type = 'UNKNOWN'
    name = None
    features = json_data.get('FEATURES', [])
    for feat in features:
        if 'RECORD_TYPE' in feat:
            record_type = feat['RECORD_TYPE']
        if not name:
            name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
    
    # Fallback to top-level fields
    if not name:
        name = json_data.get('PRIMARY_NAME_FULL')
    if not name:
        for name_obj in json_data.get('NAMES', []):
            name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
            if name:
                break
    if not name:
        name = f"Entity {entity_id}"
    
    data_sources = list(set([r.get('DATA_SOURCE') for r in records]))
    is_cross_source = len(data_sources) > 1
    
    entity_info[entity_id] = {
        'name': name,
        'type': record_type,
        'num_records': len(records),
        'is_cross_source': is_cross_source,
        'data_sources': data_sources
    }
    
    tooltip_parts = [
        name,
        f"Type: {record_type}",
        f"Entity ID: {entity_id}",
        f"Records merged: {len(records)}",
        f"Data sources: {', '.join(data_sources)}"
    ]
    if is_cross_source:
        tooltip_parts.append("CROSS-SOURCE RESOLUTION")
    tooltip = "\n".join(tooltip_parts)
    
    display_label = name[:30] + "..." if len(name) > 30 else name
    
    entity_node_id = f"entity_{entity_id}"
    G_true_combined.add_node(
        entity_node_id,
        label=display_label,
        title=tooltip,
        node_type='entity',
        type=record_type,
        num_records=len(records),
        is_cross_source=is_cross_source,
        entity_id=entity_id
    )
    entities_added += 1
    
    # Add record nodes and connect to entity
    for rec in records:
        rec_id = rec.get('RECORD_ID')
        rec_source = rec.get('DATA_SOURCE')
        rec_json = rec.get('JSON_DATA', {})
        
        # Get record name from FEATURES (v4)
        rec_name = None
        rec_type = 'UNKNOWN'
        for feat in rec_json.get('FEATURES', []):
            if 'RECORD_TYPE' in feat:
                rec_type = feat['RECORD_TYPE']
            if not rec_name:
                rec_name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
        if not rec_name:
            rec_name = name
        
        rec_tooltip = "\n".join([
            f"Record: {rec_name}",
            f"Source: {rec_source}",
            f"Type: {rec_type}",
            f"Record ID: {rec_id}",
            f"Resolves to: {name}"
        ])
        
        rec_label = rec_name[:20] + "..." if len(rec_name) > 20 else rec_name
        
        record_node_id = f"record_{rec_source}_{rec_id}"
        G_true_combined.add_node(
            record_node_id,
            label=rec_label,
            title=rec_tooltip,
            node_type='record',
            data_source=rec_source,
            type=rec_type
        )
        records_added += 1
        
        G_true_combined.add_edge(
            record_node_id,
            entity_node_id,
            edge_type='resolution'
        )

print(f"Added {entities_added} entity nodes")
print(f"Added {records_added} record nodes")
print(f"Added {G_true_combined.number_of_edges()} resolution edges")

# Add relationship edges from Senzing's RELATED_ENTITIES
relationship_edges = 0
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    
    for rel in entity.get('RELATED_ENTITIES', []):
        related_id = rel.get('ENTITY_ID')
        match_key = rel.get('MATCH_KEY', 'related')
        
        if entity_id < related_id:
            anchor_node = f"entity_{entity_id}"
            target_node = f"entity_{related_id}"
            if G_true_combined.has_node(target_node):
                G_true_combined.add_edge(
                    anchor_node,
                    target_node,
                    edge_type='relationship',
                    relationship=match_key
                )
                relationship_edges += 1

print(f"Added {relationship_edges} relationship edges")
print(f"\nTrue Combined Graph Statistics:")
print(f"  Total nodes: {G_true_combined.number_of_nodes()}")
print(f"  Total edges: {G_true_combined.number_of_edges()}")
print(f"  Entity nodes: {entities_added}")
print(f"  Record nodes: {records_added}")
print(f"  Resolution edges: {G_true_combined.number_of_edges() - relationship_edges}")
print(f"  Relationship edges: {relationship_edges}")

# Close Senzing connection
grpc_channel.close()
print("\nSenzing connection closed")

Added 196 entity nodes
Added 282 record nodes
Added 282 resolution edges
Added 492 relationship edges

True Combined Graph Statistics:
  Total nodes: 478
  Total edges: 774
  Entity nodes: 196
  Record nodes: 282
  Resolution edges: 282
  Relationship edges: 492

Senzing connection closed


## Load the Embedding Model

Loads the `all-MiniLM-L6-v2` model for creating embeddings from the raw record text. This is the same model used in the entity-resolved RAG notebook so that any differences in answers come from the data preparation, not the embedding model.

In [4]:
# Load the same embedding model used in the entity-resolved RAG notebook
print("Loading embedding model...")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


## Load Pre-Created Embeddings from LanceDB

Connects to the LanceDB instance and opens the `entities` table that was populated in notebook 07. No embeddings are created here — we reuse the pre-computed vectors.

In [5]:
db = lancedb.connect('/workspace/lancedb_data')
table = db.open_table('entities')

print(f"Connected to LanceDB")
print(f"Total entities available: {table.count_rows()}")

# Preview the pre-created embeddings
print("\nSample data from LanceDB:")
sample = table.to_pandas().head(5)
display_columns = ['entity_id', 'name', 'type', 'data_sources', 'num_records', 'risks']
print(sample[display_columns].to_string())

Connected to LanceDB
Total entities available: 196

Sample data from LanceDB:
   entity_id                               name          type                   data_sources  num_records               risks
0     100001                    Abassin Badshah        PERSON  OPEN-SANCTIONS,OPEN-OWNERSHIP            3        corp.disqual
1     100002                        LMAR GB LTD  ORGANIZATION                 OPEN-SANCTIONS            1                    
2     100003            WANDLE HOLDINGS LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1     sanction.linked
3     100004  POLYUS GOLD INTERNATIONAL LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1     sanction.linked
4     100005          Firuza Nazimovna Kerimova        PERSON                 OPEN-SANCTIONS            1  role.rca; sanction


## Configure DSPy Language Models

Sets up both Anthropic (Claude) and OpenAI (GPT) as DSPy language model backends. The `provider` parameter in the RAG function will select which one to use by reconfiguring DSPy at query time.

In [6]:
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found in environment")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment")

# Pre-build both DSPy language models
lm_anthropic = dspy.LM(
    model='anthropic/claude-sonnet-4-5-20250929',
    api_key=ANTHROPIC_API_KEY,
    max_tokens=2048
)

lm_openai = dspy.LM(
    model='openai/gpt-5.4-nano',
    api_key=OPENAI_API_KEY,
    max_tokens=2048
)

LM_REGISTRY = {
    "anthropic": lm_anthropic,
    "openai": lm_openai,
}

# Default to Anthropic
dspy.settings.configure(lm=lm_anthropic)

print("DSPy language models configured:")
print("  anthropic: Claude 4.5 Sonnet")
print("  openai:    GPT-5.4 Nano")

DSPy language models configured:
  anthropic: Claude 4.5 Sonnet
  openai:    GPT-5.4 Nano


## Define the DSPy Signature and Module

Defines the contract for the RAG pipeline using a DSPy `Signature`: the LLM receives a `context` field (entity and relationship data from the knowledge graph) and a `question` field, and must produce an `answer`. The `KnowledgeGraphRAG` module wraps this in a `ChainOfThought` call, which prompts the model to reason step by step before answering rather than jumping straight to a conclusion.

In [7]:
# Define DSPy signature for Knowledge Graph RAG
class KnowledgeGraphRAGSignature(dspy.Signature):
    """Answer questions about a corporate ownership and sanctions knowledge graph."""
    
    context = dspy.InputField(desc="Entity and relationship data from the corporate ownership and sanctions knowledge graph")
    question = dspy.InputField(desc="User's question about the data")
    answer = dspy.OutputField(desc="Detailed answer based on the retrieved entities and their relationships")

# Create DSPy module
class KnowledgeGraphRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(KnowledgeGraphRAGSignature)
    
    def forward(self, context, question):
        return self.generate_answer(context=context, question=question)

# Initialize
kg_rag = KnowledgeGraphRAG()

## RAG Query Function

Defines the RAG pipeline: vector search in LanceDB, neighbor expansion via the knowledge graph, context assembly, and DSPy-powered LLM query with chain-of-thought reasoning. The `provider` parameter controls which LLM backend is used — `"anthropic"` for Claude or `"openai"` for GPT.

In [8]:
def ask_knowledge_graph(question, provider="anthropic", top_k=10):
    """
    Graph-augmented RAG over entity-resolved data via DSPy (ChainOfThought).

    1. Vector search to find top-k relevant entities
    2. Expand to graph neighbors (1 hop) for relationship context
    3. Build context with entity details and relationship info
    4. Query the selected LLM via DSPy

    Parameters
    ----------
    question : str
        The user's question.
    provider : str
        LLM backend — "anthropic" (Claude) or "openai" (GPT).
    top_k : int
        Number of entities to retrieve via vector search.
    """
    if provider not in ("anthropic", "openai"):
        raise ValueError(f"provider must be 'anthropic' or 'openai', got '{provider}'")

    # Switch DSPy to the selected provider
    dspy.settings.configure(lm=LM_REGISTRY[provider])

    print(f"\nQuestion: {question}")
    print(f"Provider: {provider}")
    print("=" * 70)

    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()

    print(f"Found {len(results)} relevant entities")

    # Step 2: Collect entity IDs and expand to neighbors via G_true_combined
    entity_ids = set()
    for r in results:
        entity_ids.add(r['entity_id'])

        entity_node = f"entity_{r['entity_id']}"
        if entity_node in G_true_combined:
            for neighbor in list(G_true_combined.neighbors(entity_node))[:8]:
                nd = G_true_combined.nodes[neighbor]
                # Follow relationship edges to other entities
                if nd.get('node_type') == 'entity':
                    entity_ids.add(nd.get('entity_id'))

    print(f"Expanded to {len(entity_ids)} entities (including neighbors)")

    # Step 3: Build context with graph structure
    context_parts = ["ENTITIES:\n"]

    for entity_id in list(entity_ids)[:30]:
        ei = table.search().where(f"entity_id = {entity_id}").limit(1).to_list()
        if not ei:
            continue

        info = ei[0]
        context_parts.append(f"- {info['name']} ({info['type']})")
        context_parts.append(f"  Sources: {info['data_sources']}, Records: {info['num_records']}")

        if info.get('risks'):
            context_parts.append(f"  Risks: {info['risks']}")

        # Get relationships and record info from G_true_combined
        entity_node = f"entity_{entity_id}"
        if entity_node in G_true_combined:
            rels = []
            record_sources = []
            for neighbor in G_true_combined.neighbors(entity_node):
                nd = G_true_combined.nodes[neighbor]
                edge_data = G_true_combined.get_edge_data(entity_node, neighbor)

                if nd.get('node_type') == 'entity' and edge_data.get('edge_type') == 'relationship':
                    rel_type = edge_data.get('relationship', 'connected to')
                    rels.append(f"{rel_type} {nd.get('label', 'Unknown')}")
                elif nd.get('node_type') == 'record':
                    record_sources.append(nd.get('data_source', 'unknown'))

            if record_sources:
                context_parts.append(f"  Source records: {', '.join(record_sources)}")
            if rels:
                context_parts.append(f"  Relationships: {'; '.join(rels[:5])}")

        context_parts.append("")

    context = "\n".join(context_parts)

    # Step 4: Ask LLM via DSPy
    print("Querying LLM...")

    result = kg_rag(context=context, question=question)

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(result.answer)
    print("=" * 70)

## Interactive Chatbot Session

Runs a simple read-eval-print loop so participants can ask free-form questions about the dataset. Type `quit` or `exit` to stop. This version uses DSPy with chain-of-thought reasoning over the entity-resolved knowledge graph. Change `provider` below to switch between Claude and GPT.

In [9]:
provider = "anthropic"  # change to "openai" to use GPT-5.4 nano

print("Knowledge Graph RAG (DSPy) - Interactive Session")
print("=" * 70)
print("Ask any question about the corporate ownership and sanctions data.")
print("The system will search LanceDB and query the knowledge graph.")
print(f"Current LLM provider: {provider}")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()

    if question.lower() in ('quit', 'exit', 'q'):
        print("Goodbye!")
        break

    if not question:
        continue

    try:
        ask_knowledge_graph(question, provider=provider)
        print()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Knowledge Graph RAG (DSPy) - Interactive Session
Ask any question about the corporate ownership and sanctions data.
The system will search LanceDB and query the knowledge graph.
Current LLM provider: anthropic
Type 'quit' to exit.



Your question:  tell me about said kerimov



Question: tell me about said kerimov
Provider: anthropic
Found 10 relevant entities
Expanded to 22 entities (including neighbors)
Querying LLM...

ANSWER
Said Kerimov is a sanctioned individual who appears in both sanctions databases and corporate ownership records, indicating significant international scrutiny.

**Sanctions and Risk Profile:**
Said Kerimov is designated as a sanctioned person and classified as a "related/close associate" (role.rca). He is part of a family network where all members are under sanctions, including his apparent relatives: Firuza Nazimovna Kerimova, Amina Suleymanovna Kerimova, and Gulnara Suleimanova KERIMOVA. Most significantly, he has family connections to Suleyman Abusaidovich KERIMOV, who is identified as an oligarch, politically exposed person (PEP), person of interest, and sanctioned individual.

**Russian Business Interests:**
Said Kerimov has connections to several Russian limited liability companies (ООО), all of which are flagged as sanction-li

Your question:  q


Goodbye!
